### Choose where this notebook runs

This is the **IQM edition**: the 2-qubit and 3-qubit Grover searches (and the circuits of the In-depth
analysis) are transpiled to the IQM native gate set **PRX + CZ** and run on **Garnet** (20 qubits) or
**Emerald** (54 qubits) via IQM Resonance. Pick the backend by editing the single `TARGET` line below
(see `README.md` and `iqm_backend.py` in this folder):

| `TARGET` | runs on | token? | QPU credits? |
|---|---|---|---|
| `"fake_apollo"` | local 20-qubit noisy simulator | no | no |
| `"garnet:mock"` / `"emerald:mock"` | IQM Resonance cloud mock (pipeline check: returns RANDOM bitstrings, no physics) | yes | no |
| `"garnet"` / `"emerald"` | **real QPU** (20q / 54q) | yes | yes |


# Grover's Search Algorithm

**Course**: Quantum Computing Laboratory  
**Version**: 2.2 (Qiskit 2.2)  
**Last Updated**: November 2025

---

## 🎯 Quantum Search in √N Steps

Grover's algorithm provides **quadratic speedup** for unstructured search:
- Classical: O(N) queries
- Quantum: O(√N) queries

In [ ]:
from iqm_backend import get_backend, run_counts   # helper module in this folder (see iqm_backend.py)
# run_counts(circuit, TARGET, shots=...) = transpile to the IQM native gates (PRX + CZ) + run + get_counts()
# get_backend(TARGET) returns the raw Qiskit backend object if you want to call backend.run yourself
TARGET = "fake_apollo"   # change this one line to choose the backend (see the table above)


## 📚 Theory

### The Problem
Given N items, find the ONE marked item.

**Classical**: Check N/2 items on average  
**Quantum**: Check √N items!

### How It Works
1. **Superposition**: Create equal amplitudes for all states
2. **Oracle**: Mark the target (phase flip)
3. **Diffusion**: Amplify marked amplitude
4. **Repeat**: ~√N times
5. **Measure**: Find target with high probability

In [ ]:
from qiskit import QuantumCircuit                                   # the object we build circuits with
from qiskit.circuit.library import grover_operator                  # library Grover iterate (oracle + diffusion); Qiskit >= 2.1 name of the former GroverOperator class
from qiskit.quantum_info import Statevector, Operator               # exact state vector / exact unitary matrix of a circuit (checks and In-depth analysis)
# transpilation + execution are done by run_counts (imported above) on the IQM backend chosen by TARGET
from qiskit.visualization import plot_histogram                     # bar chart of measurement counts
import numpy as np                                                  # arrays and maths (np.pi, np.sin, np.arcsin)
import matplotlib.pyplot as plt                                     # plotting (success probability vs iterations, in the In-depth analysis)
from IPython.display import display                      # show a figure (e.g. a circuit drawing) from the middle of a cell
def draw(qc, fold=-1, **kw): return qc.draw("mpl", style="clifford", fold=fold, **kw)   # every circuit picture: matplotlib drawer; fold=-1 = one long row, fold=N wraps after N columns

import qiskit; print(f'✓ Qiskit {qiskit.__version__} ready | TARGET = {TARGET}')


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `grover_operator(oracle)` | builds the whole Grover iterate (oracle followed by the diffusion) from any oracle circuit; it replaces the `GroverOperator` class (deprecated since Qiskit 2.1) and is used in the In-depth analysis to cross-check the hand-built version |
| `Statevector`, `Operator` | the exact state vector / unitary matrix of a circuit: how we verify by hand what the backend samples |
| `run_counts`, `TARGET` (cell above) | execution goes through the folder helper: transpile to the native PRX + CZ gates, run on the backend chosen by `TARGET`, return {bitstring: count} — no Aer, no Sampler in this edition |
| `display`, `draw(qc)` | our one-line helper around `qc.draw("mpl", style="clifford")`; `display()` shows a figure from the middle of a cell |

</details>


## 🔨 Implementation: 2-Qubit Example

### Oracle: Mark |11⟩

In [ ]:
def oracle_2q():
    """Oracle marks |11⟩"""
    qc = QuantumCircuit(2)                        # 2 qubits, no classical bits: a building block to be composed into a bigger circuit
    qc.cz(0, 1)  # Phase flip on |11⟩: controlled-Z multiplies the amplitude of |11⟩ by −1 and leaves the other three basis states untouched
    return qc

# Diffusion operator
def diffusion_2q():
    """Inversion about average"""
    qc = QuantumCircuit(2)
    qc.h([0, 1])                                  # a list of qubits applies the gate to each of them: H⊗H maps the uniform state |s⟩ to |00⟩
    qc.z([0, 1])                                  # Z⊗Z = diag(1, −1, −1, 1) ...
    qc.cz(0, 1)                                   # ... times CZ = diag(1, 1, 1, −1) gives diag(1, −1, −1, −1) = 2|00⟩⟨00| − I: a reflection about |00⟩
    qc.h([0, 1])                                  # back to the Hadamard basis: altogether 2|s⟩⟨s| − I, the reflection about the uniform state
    return qc

print('✓ Oracle and diffusion defined')
display(draw(oracle_2q())); print("↑ the oracle for |11⟩: a single CZ — the only basis state with both qubits at 1 gets the sign −1")
display(draw(diffusion_2q())); print("↑ the diffusion operator: H⊗H · (Z⊗Z · CZ) · H⊗H = 2|s⟩⟨s| − I, the 'inversion about the average'")
print("matrix of the diffusion operator (rows/columns ordered 00, 01, 10, 11):")
print(np.round(Operator(diffusion_2q()).data.real, 2))             # Operator(circuit).data = the unitary as a NumPy array (real here)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `QuantumCircuit(2)` | 2 qubits and no classical bits: a block to be composed into a bigger circuit later |
| `qc.cz(0, 1)` | the oracle: −1 on \|11⟩ only — the 'marking' is a phase, invisible to a measurement until the diffusion turns it into amplitude |
| `qc.h([0, 1])`, `qc.z([0, 1])` | a list applies the same gate to every qubit listed |
| `H·(Z⊗Z·CZ)·H` | Z⊗Z·CZ = diag(1, −1, −1, −1) = 2\|00⟩⟨00\| − I; conjugated by H⊗H it becomes 2\|s⟩⟨s\| − I: every amplitude a → 2ā − a, the inversion about the average ā |
| `Operator(circ).data` | the 4×4 matrix of the circuit (rows/columns 00, 01, 10, 11): −1/2 on the diagonal, +1/2 elsewhere, as worked out in the box below |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**The two 2-qubit unitaries by hand.** Basis order $|00\rangle, |01\rangle, |10\rangle, |11\rangle$.

*Oracle.* $\mathrm{CZ} = \mathrm{diag}(1, 1, 1, -1)$: it multiplies the $|11\rangle$ amplitude by $-1$ and does nothing else.

*Diffusion.* $Z\otimes Z = \mathrm{diag}(1,-1,-1,1)$ and $\mathrm{CZ} = \mathrm{diag}(1,1,1,-1)$, so $Z\otimes Z\cdot \mathrm{CZ} = \mathrm{diag}(1,-1,-1,-1) = 2|00\rangle\langle 00| - I$. Sandwiched between two $H\otimes H$ (which send $|00\rangle$ to the uniform state $|s\rangle = \tfrac12(1,1,1,1)^T$):

$$D = H^{\otimes 2}\,(2|00\rangle\langle 00| - I)\,H^{\otimes 2} = 2|s\rangle\langle s| - I = \begin{pmatrix} -\tfrac12 & \tfrac12 & \tfrac12 & \tfrac12 \\ \tfrac12 & -\tfrac12 & \tfrac12 & \tfrac12 \\ \tfrac12 & \tfrac12 & -\tfrac12 & \tfrac12 \\ \tfrac12 & \tfrac12 & \tfrac12 & -\tfrac12 \end{pmatrix},$$

the matrix the cell prints. On amplitudes $(a_0, a_1, a_2, a_3)$ with average $\bar a$ it acts as $a_i \to 2\bar a - a_i$: the *inversion about the average*.

*One iteration on $|s\rangle$.* The oracle gives $\tfrac12(1, 1, 1, -1)$, whose average is $\bar a = \tfrac14$; the inversion maps $\tfrac12 \to 2\cdot\tfrac14 - \tfrac12 = 0$ and $-\tfrac12 \to \tfrac12 + \tfrac12 = 1$. The state is exactly $|11\rangle$: on 2 qubits a single Grover iteration succeeds with probability $1$, which is what the execution cell below measures.

</details>


### Complete Grover Circuit

In [ ]:
# Build circuit
qc = QuantumCircuit(2, 2)                          # QuantumCircuit(number_of_qubits, number_of_classical_bits)

# Initialize: superposition
qc.h([0, 1])                                       # |00⟩ -> |s⟩ = (|00⟩ + |01⟩ + |10⟩ + |11⟩)/2: every item of the 'database' has amplitude 1/2
qc.barrier()                                       # a visual separator in the drawing (and a fence the transpiler does not optimise across)

# Grover iteration (1 iteration for N=4)
qc.compose(oracle_2q(), inplace=True)              # compose: append another circuit's gates; inplace=True modifies qc instead of returning a copy
qc.barrier()
qc.compose(diffusion_2q(), inplace=True)
qc.barrier()

# Measure
qc.measure([0, 1], [0, 1])                         # measure(qubits, classical_bits): qubit i -> bit i

print(f'Circuit depth: {qc.depth()}')              # depth = number of layers of gates that cannot run in parallel (barriers are not counted)
print(f'Gates: {qc.size()}')                       # size = number of instructions, measurements included, barriers excluded
display(draw(qc)); print("↑ one Grover iteration on 2 qubits: H⊗H (superposition) | oracle CZ | diffusion H·Z·CZ·Z·H | measurement")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `QuantumCircuit(2, 2)` | 2 qubits + 2 classical bits (a register called `c`) for the result |
| `qc.h([0, 1])` | the uniform superposition \|s⟩: amplitude 1/2 on each of the N = 4 items — 'the database' |
| `qc.compose(block, inplace=True)` | glue a sub-circuit onto qc; without `inplace=True` compose returns a new circuit and leaves qc unchanged |
| `qc.barrier()` | a dashed line that separates the stages in the drawing and stops the transpiler from merging gates across it |
| `qc.depth()`, `qc.size()` | 7 layers and 12 instructions (barriers are not counted; the two measurements are) |
| `display(draw(qc))` | our helper; the picture shows the three stages of one Grover iteration between the barriers |

</details>


### Execute and Results

In [ ]:
# Run on the IQM backend chosen by TARGET
# run_counts = transpile_to_IQM (native PRX + CZ gates, routed on the chip's square lattice) + backend.run + get_counts.
# return_transpiled=True also returns the transpiled circuit, so we can inspect what the hardware will execute.
counts, transpiled = run_counts(qc, TARGET, shots=1024, return_transpiled=True)
print(f"transpiled for IQM: {transpiled.size()} native instructions, depth {transpiled.depth()}, {dict(transpiled.count_ops())}")   # 'r' = PRX rotation, 'cz' = controlled-Z

def run_circuit(circuit, shots=1024):
    """Transpile + run on TARGET + counts in one call (used by the In-depth analysis at the end)."""
    return run_counts(circuit, TARGET, shots=shots)                       # counts = plain dict {bitstring: count}

from iqm.qiskit_iqm import transpile_to_IQM                                # the IQM transpiler alone, to look at a circuit without running it
def to_native(circuit):
    """Rewrite a circuit in the native gates of IQM hardware, {r (PRX), cz}, on TARGET's qubit lattice: what the chip executes."""
    return transpile_to_IQM(circuit, backend=get_backend(TARGET), optimization_level=3)

print('Results:')
for state, count in sorted(counts.items(), key=lambda x: -x[1]):          # the (bitstring, count) pairs by decreasing count
    print(f'  |{state}⟩: {count} ({count/1024:.1%})')                     # :.1% formats a fraction as a percentage with one decimal

plot_histogram(counts)                                                     # the last expression of a cell is displayed: here the bar chart


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `run_counts(qc, TARGET, shots=1024, return_transpiled=True)` | the helper's full pipeline: `transpile_to_IQM` (native r = PRX and cz, routed on the chip's square lattice) → `backend.run` → counts; the transpiled circuit comes back too |
| `'r'`, `'cz'` in `count_ops()` | IQM's two native gates: every H, Z and CZ of ours became some of them; `size()` must stay below the 10 000-instruction cap of IQM Resonance (here ten) |
| `TARGET` | `fake_apollo` = local noisy 20-qubit model (no token); `garnet:mock` = cloud pipeline check that returns RANDOM bitstrings; `garnet` / `emerald` = the real QPUs (token + credits) |
| `run_circuit`, `to_native` | the same call in one function, and `transpile_to_IQM` alone (to inspect a circuit without running it) — both used by the In-depth analysis at the end |
| `counts` | a plain dict {bitstring: count}; on `fake_apollo` about 15 % of the shots leak to \|01⟩, \|10⟩ and \|00⟩: gate and readout noise, absent from the ideal 100 % of the box below |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**The geometric picture of one iteration.** Split the uniform state into the marked part $|w\rangle$ ($M$ states) and the unmarked part $|w^\perp\rangle$ ($N-M$ states):

$$|s\rangle = \sin\theta\,|w\rangle + \cos\theta\,|w^\perp\rangle, \qquad \sin\theta = \sqrt{M/N}.$$

The oracle is a reflection about $|w^\perp\rangle$ (it flips the sign of the $|w\rangle$ component); the diffusion $2|s\rangle\langle s| - I$ is a reflection about $|s\rangle$. Two reflections make a rotation by twice the angle between the mirrors, $2\theta$, towards $|w\rangle$: after $k$ iterations the state is

$$\sin\big((2k+1)\theta\big)\,|w\rangle + \cos\big((2k+1)\theta\big)\,|w^\perp\rangle, \qquad P_{\rm success}(k) = \sin^2\big((2k+1)\theta\big).$$

**Optimal number of iterations.** We want $(2k+1)\theta \approx \pi/2$, i.e.

$$k_{\rm opt} = \operatorname{round}\!\Big(\frac{\pi}{4\theta} - \frac12\Big) \;\approx\; \frac{\pi}{4}\sqrt{\frac{N}{M}} \qquad (\theta \approx \sin\theta \text{ for large } N),$$

the $O(\sqrt N)$ of the introduction. More iterations are *not* better: the state rotates past $|w\rangle$ and the probability goes down again (task 1 of the In-depth analysis).

**Why one iteration is exact on 2 qubits.** $N = 4$, $M = 1$: $\sin\theta = \tfrac12$, $\theta = 30^\circ$, so after one iteration the angle is $3\theta = 90^\circ$ — the state *is* $|w\rangle = |11\rangle$ and $P = \sin^2 90^\circ = 1$. The formula gives $k_{\rm opt} = \operatorname{round}(1.5 - 0.5) = 1$ (the rough $\tfrac{\pi}{4}\sqrt{4} \approx 1.57$ of the Analysis cell rounds to the same). Hence the 1024 of 1024 shots on $|11\rangle$ printed above by the ideal simulator — a noisy device loses a fraction of them (the IQM edition on `fake_apollo`: about 85 % on $|11\rangle$).

</details>


## 📊 Analysis

**Expected**: |11⟩ with ~100% probability  
**Why**: One iteration is optimal for N=4  

### Optimal Iterations
For N items with M solutions:
$$\text{iterations} \approx \frac{\pi}{4}\sqrt{\frac{N}{M}}$$

For N=4, M=1: π/4 × √4 = π/4 × 2 ≈ 1.57 ≈ 1 iteration

## 🎓 3-Qubit Example

### Oracle: Mark |101⟩

In [ ]:
def oracle_3q_mark_101():
    """Mark |101⟩"""
    qc = QuantumCircuit(3)
    # Apply X to qubit 1 (middle qubit is 0 in |101⟩)
    qc.x(1)                                       # relabel: |101⟩ -> |111⟩ (Qiskit writes the bitstring q2 q1 q0, so the middle 0 is qubit 1)
    # Multi-controlled Z
    qc.h(2)                                       # H · CCX · H on the target qubit = CCZ: −1 on |111⟩ only
    qc.ccx(0, 1, 2)                               # Toffoli: controls 0 and 1, target 2
    qc.h(2)
    qc.x(1)                                       # undo the relabelling: net effect −1 on |101⟩ only
    return qc

def diffusion_3q():
    """Diffusion for 3 qubits"""
    qc = QuantumCircuit(3)
    qc.h([0, 1, 2])                               # |s⟩ -> |000⟩
    qc.x([0, 1, 2])                               # |000⟩ -> |111⟩ ...
    qc.h(2)
    qc.ccx(0, 1, 2)                               # ... so that the CCZ (−1 on |111⟩) flips the sign of what was |000⟩
    qc.h(2)
    qc.x([0, 1, 2])
    qc.h([0, 1, 2])                               # altogether −(2|s⟩⟨s| − I): the reflection about |s⟩ (the global sign is irrelevant)
    return qc

# Build 3-qubit circuit
qc3 = QuantumCircuit(3, 3)
qc3.h([0, 1, 2])                                  # uniform superposition of the N = 8 items
qc3.barrier()

# 2 iterations for N=8
for _ in range(2):                                # _ = conventional name for a loop variable we do not use
    qc3.compose(oracle_3q_mark_101(), inplace=True)
    qc3.barrier()
    qc3.compose(diffusion_3q(), inplace=True)
    qc3.barrier()

qc3.measure([0, 1, 2], [0, 1, 2])
print(f'Circuit depth: {qc3.depth()}, gates: {qc3.size()}')
display(draw(oracle_3q_mark_101())); print("↑ the oracle for |101⟩: X on qubit 1 relabels the target to |111⟩, H·CCX·H on qubit 2 is a CCZ, the second X undoes the relabelling")
display(draw(diffusion_3q())); print("↑ the 3-qubit diffusion: H⊗3 · X⊗3 · CCZ · X⊗3 · H⊗3 — a reflection about |s⟩ built from a reflection about |000⟩")
display(draw(qc3, fold=25)); print("↑ the full search: superposition, then (oracle | diffusion) twice, then the measurement; 25 columns per row")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qc.x(1)` … `qc.x(1)` | the target \|101⟩ has a 0 in the middle (qubit 1, since Qiskit writes the bitstring q2 q1 q0): X relabels it to \|111⟩ and is undone afterwards |
| `qc.h(2); qc.ccx(0, 1, 2); qc.h(2)` | a Toffoli between two Hadamards on its target is a CCZ: −1 on \|111⟩ only — the 3-qubit phase oracle |
| `diffusion_3q` | H⊗3 · X⊗3 · CCZ · X⊗3 · H⊗3 = −(2\|s⟩⟨s\| − I): the X's turn the flip of \|111⟩ into a flip of \|000⟩, the H's turn \|000⟩ into \|s⟩ (the global sign is irrelevant) |
| `for _ in range(2)` | two Grover iterations: with θ = arcsin(1/√8) = 20.7° the optimum (box after the execution cell); `_` is the conventional name for an unused loop variable |
| `qc3.measure([0, 1, 2], [0, 1, 2])` | qubit i → bit i; `draw(qc3, fold=25)` wraps the long picture after 25 columns |

</details>


### Execute 3-Qubit Search

In [ ]:
counts3, transpiled3 = run_counts(qc3, TARGET, shots=1024, return_transpiled=True)   # the 3-qubit search on TARGET
print(f"transpiled for IQM: {transpiled3.size()} native instructions, depth {transpiled3.depth()}, {dict(transpiled3.count_ops())}")   # 4 Toffolis -> dozens of cz (+ routing swaps)

print('3-Qubit Results:')
for state, count in sorted(counts3.items(), key=lambda x: -x[1])[:3]:    # [:3] = a slice: only the three most frequent outcomes
    print(f'  |{state}⟩: {count} ({count/1024:.1%})')

plot_histogram(counts3)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `run_counts(qc3, TARGET, shots=1024, return_transpiled=True)` | the 3-qubit search on the IQM backend: 4 Toffolis → dozens of cz plus the r rotations and the swaps the router inserts (a Toffoli needs its three qubits pairwise coupled, which the square lattice does not offer) |
| `transpiled3.depth()` | the depth of the native circuit sets its duration, hence the decoherence; compare with the 22 layers of the abstract circuit (barriers excluded) |
| `sorted(...)[:3]` | a slice: only the three most frequent outcomes are printed |
| `plot_histogram(counts3)` | on `fake_apollo` the \|101⟩ peak drops from the ideal 94.5 % to roughly a third of the shots and the seven other states fill up: ~40 CZ gates' worth of noise, a first measure of what a real run costs |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**Three qubits, one marked state.** $N = 8$, $M = 1$: $\sin\theta = 1/\sqrt8$, $\theta = 20.7^\circ$. The success probability after $k$ iterations, $\sin^2\big((2k+1)\theta\big)$:

| $k$ | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| angle $(2k+1)\theta$ | 20.7° | 62.1° | 103.5° | 144.9° | 186.3° |
| $P(\lvert 101\rangle)$ | 0.125 | 0.781 | **0.945** | 0.330 | 0.012 |

$k_{\rm opt} = \operatorname{round}(\pi/(4\theta) - \tfrac12) = \operatorname{round}(2.17 - 0.5) = 2$ ✓ (the rough $\tfrac\pi4\sqrt8 = 2.22$ also rounds to 2), and a third iteration would *overshoot* to 33 %. Unlike the 2-qubit case the angle $5\theta = 103.5^\circ$ does not hit $90^\circ$ exactly, so the ideal simulator prints $\approx 94.5\,\%$ for $|101\rangle$ — $968 \pm 7$ of 1024 shots — and leaves $5.5\,\%$ spread uniformly over the other seven states, $\approx 8$ shots each, which is what the two runners-up show. On a noisy device (the IQM edition on `fake_apollo`: four Toffolis, each several CZ gates plus routing, ~40 CZ gates in all) the peak drops to roughly a third of the shots and the background rises accordingly.

</details>


## 💡 Key Insights

### Complexity Comparison
| Database Size (N) | Classical Queries | Quantum Queries | Speedup |
|------------------|-------------------|-----------------|----------|
| 4 | 2 | 1 | 2× |
| 16 | 8 | 2 | 4× |
| 256 | 128 | 8 | 16× |
| 1,000,000 | 500,000 | 500 | 1000× |

### Limitations
- Speedup is quadratic, not exponential
- Requires knowing number of solutions
- Needs amplitude amplification (delicate)
- Best for unstructured search

## 🎓 Exercises

### Exercise 1: Different Target (Easy)
Modify oracle to search for |10⟩ in 2-qubit example.

### Exercise 2: Multiple Solutions (Medium)
Create oracle marking both |101⟩ AND |110⟩. Calculate optimal iterations.

### Exercise 3: 4-Qubit Search (Hard)
Implement complete Grover for 4 qubits, marking |1010⟩.

## 🔬 In-depth analysis

1. **Success probability vs number of iterations.** For the 3-qubit search of $|101\rangle$, build the circuit with $k = 0, 1, \dots, 6$ Grover iterations, measure the probability of $|101\rangle$ with `run_circuit` and plot it against $k$ together with the prediction $\sin^2\big((2k+1)\theta\big)$, $\theta = \arcsin(1/\sqrt8)$ (and, if you like, the exact value from `Statevector`). Where is the maximum, and what happens if you 'keep amplifying' beyond it?
2. **Two marked states** (the author's Exercise 2). Build an oracle that flips the sign of both $|101\rangle$ and $|110\rangle$ (hint: one phase-flip block per marked state, composed one after the other), compute the optimal number of iterations for $M = 2$, $N = 8$ with $k = \operatorname{round}(\pi/(4\theta) - 1/2)$, run it and check that the two states share the counts. Compare with the 2-qubit case: why is one iteration again exact?
3. **The native-gate cost.** Check with `Operator(...).equiv` that the library iterate `grover_operator(oracle_3q_mark_101())` equals the author's oracle + diffusion, then rewrite the 3-qubit circuit `qc3` with `to_native` (defined in the execution cell: the native gate set of this edition's hardware) and report `count_ops()`, `depth()` and the number of two-qubit gates. How many two-qubit gates does each Toffoli cost, and what does that imply for the noise on a real device?

Write your code in the empty cell below; a reference solution follows it. Try first, then compare.


In [ ]:
# 🔬 In-depth analysis — your code here



In [ ]:
# Reference solution
# (1) success probability vs number of iterations, 3 qubits, target |101⟩
def grover_3q(k, measure=True):
    """The 3-qubit search with k Grover iterations (the circuit of the 3-Qubit Example for k = 2)."""
    circ = QuantumCircuit(3, 3)
    circ.h([0, 1, 2])
    for _ in range(k):
        circ.compose(oracle_3q_mark_101(), inplace=True)
        circ.compose(diffusion_3q(), inplace=True)
    if measure: circ.measure([0, 1, 2], [0, 1, 2])                      # measure=False: the version Statevector needs
    return circ

theta = np.arcsin(np.sqrt(1 / 8))                                        # sin θ = √(M/N) with M = 1 marked state out of N = 8
ks = np.arange(7)
p_theory = np.sin((2 * ks + 1) * theta) ** 2                             # the geometric picture: angle (2k+1)θ after k iterations
p_exact = [Statevector(grover_3q(k, measure=False)).probabilities_dict().get("101", 0.0) for k in ks]   # exact |amplitude|² from the state vector
p_meas = [run_circuit(grover_3q(k), shots=1024).get("101", 0) / 1024 for k in ks]                       # sampled on the backend; .get(key, 0) if the key is missing
for k in ks:
    print(f"(1) k = {k}: theory {p_theory[k]:.3f}, statevector {p_exact[k]:.3f}, measured {p_meas[k]:.3f}")
print(f"(1) optimal k = round(π/(4θ) − 1/2) = {round(np.pi / (4 * theta) - 0.5)}; beyond it the state rotates past |101⟩ and the probability falls again")
fig, ax = plt.subplots(figsize=(6, 3.5))
kk = np.linspace(0, 6, 200)
ax.plot(kk, np.sin((2 * kk + 1) * theta) ** 2, "-", label="sin²((2k+1)θ)")   # the continuous curve
ax.plot(ks, p_exact, "s", mfc="none", label="statevector (exact)")
ax.plot(ks, p_meas, "o", label="measured (1024 shots)")
ax.set(xlabel="Grover iterations k", ylabel="P(|101⟩)", title="3-qubit search: success probability vs iterations"); ax.legend()
display(fig); plt.close(fig); print("↑ the theory curve oscillates: peak at k = 2 (94.5 %), again near k = 6, a dip to ≈ 1 % at k = 4; the measured points follow it on an ideal simulator and sink towards 1/8 with the depth on a noisy backend")

# (2) two marked states |101⟩ and |110⟩ (the author's Exercise 2)
def oracle_3q_mark_110():
    """Mark |110⟩: X on qubit 0 (the 0 of the target is the rightmost bit), CCZ, undo the X."""
    circ = QuantumCircuit(3)
    circ.x(0); circ.h(2); circ.ccx(0, 1, 2); circ.h(2); circ.x(0)
    return circ

oracle_two = oracle_3q_mark_101().compose(oracle_3q_mark_110())          # two phase flips in a row: the diagonal has −1 at both marked states
print("(2) oracle diagonal (states 000 … 111):", np.round(np.diag(Operator(oracle_two).data).real).astype(int))
theta2 = np.arcsin(np.sqrt(2 / 8)); k2 = round(np.pi / (4 * theta2) - 0.5)
qc_two = QuantumCircuit(3, 3); qc_two.h([0, 1, 2])
for _ in range(k2):
    qc_two.compose(oracle_two, inplace=True); qc_two.compose(diffusion_3q(), inplace=True)
qc_two.measure([0, 1, 2], [0, 1, 2])
counts_two = run_circuit(qc_two, shots=1024)
top = sorted(counts_two.items(), key=lambda x: -x[1])[:2]
print(f"(2) M = 2, N = 8: sin θ = 1/2, θ = 30° exactly as for 2 qubits -> k = {k2} iteration is exact; measured {top} of 1024 shots")

# (3) library check and native-gate cost
iterate = oracle_3q_mark_101().compose(diffusion_3q())                  # the author's Grover iterate as one circuit
print("(3) grover_operator(oracle) equals oracle + diffusion (up to a global phase):",
      Operator(grover_operator(oracle_3q_mark_101())).equiv(Operator(iterate)))
native = to_native(qc3)
ops = dict(native.count_ops()); two_q = sum(v for g, v in ops.items() if g in ("cx", "cz", "ecr"))   # the two-qubit gates of either hardware family
print(f"(3) qc3 in native gates: {ops}, depth {native.depth()}")
print(f"    {two_q} two-qubit gates for 4 Toffolis -> {two_q / 4:.1f} per Toffoli (an ideal Toffoli needs 6 CNOTs; routing on a lattice adds swaps = 3 more each)")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `grover_3q(k, measure=True)` | the circuit as a function of the number of iterations; `measure=False` gives the measurement-free version that `Statevector` needs |
| `Statevector(circ).probabilities_dict()` | exact probabilities {bitstring: p} from the state vector — the theory curve without shot noise |
| `run_circuit(...).get("101", 0) / 1024` | the sampled probability; `.get(key, 0)` returns 0 if the key is missing instead of raising |
| `np.sin((2 * ks + 1) * theta) ** 2` | the geometric formula evaluated on a NumPy array: one line for all k |
| `oracle_a.compose(oracle_b)` | two phase oracles in sequence multiply their diagonals: −1 appears exactly on the union of the marked states |
| `Operator(A).equiv(Operator(B))` | equality of two unitaries up to a global phase — the right notion of 'same circuit' |
| `to_native(qc3)`, `count_ops()` | the circuit in the hardware's native gates; the two-qubit gate count (cx on IBM, cz on IQM) is the best single predictor of the error on a real device |

</details>


## 📖 References

1. Grover, L.K. (1996). *A fast quantum mechanical algorithm for database search*. STOC '96.
2. Nielsen & Chuang, *Quantum Computation and Quantum Information*
3. Qiskit Textbook: https://learn.qiskit.org

---

**Version**: 2.2.0  
**License**: MIT